In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_tiny import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 15,710
  -> ny bästa modell sparad till ../models/cnn_tiny.pth
Epoch   0 | train: 0.8461 | val: 0.3946 | acc: 92.39% | AUC: 0.920  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_tiny.pth
Epoch   1 | train: 0.4463 | val: 0.2795 | acc: 95.83% | AUC: 0.963  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_tiny.pth
Epoch   2 | train: 0.3421 | val: 0.2490 | acc: 94.92% | AUC: 0.978  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_tiny.pth
Epoch   3 | train: 0.2884 | val: 0.1592 | acc: 98.14% | AUC: 0.984  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_tiny.pth
Epoch   4 | train: 0.2460 | val: 0.1350 | acc: 98.11% | AUC: 0.991  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_tiny.pth
Epoch   5 | train: 0.2248 | val: 0.1727 | acc: 96.71% | AUC: 0.994  | LR: 0.001
Epoch   6 | train: 0.1989 | val: 0.1573 | acc: 96.93% | AUC: 0.994  |

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.096663,0.158665,96.193129,0.981837,0.969214,0.911111,1826,58,24,246
1,2,0.159428,0.192048,92.711235,0.987351,0.921975,0.962963,1737,147,10,260
2,3,0.102233,0.195060,95.589601,0.978423,0.969231,0.862454,1827,58,37,232
3,4,0.073041,0.194257,97.396560,0.981717,0.981403,0.921933,1847,35,21,248
4,5,0.103076,0.173158,95.169531,0.983121,0.955414,0.925651,1800,84,20,249
5,6,0.120821,0.121275,96.189591,0.990747,0.963356,0.951673,1814,69,13,256
6,7,0.092831,0.185254,97.770553,0.981825,0.992569,0.873606,1870,14,34,235
7,8,0.084634,0.267743,96.703807,0.976472,0.985676,0.836431,1858,27,44,225
8,9,0.140555,0.294323,93.268338,0.963748,0.936340,0.907063,1765,120,25,244
9,10,0.095692,0.133943,95.304510,0.988618,0.953773,0.947955,1795,87,14,255


In [2]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_small import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 45,266
  -> ny bästa modell sparad till ../models/cnn_small.pth
Epoch   0 | train: 0.9604 | val: 0.4805 | acc: 90.08% | AUC: 0.909  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_small.pth
Epoch   1 | train: 0.4255 | val: 0.2241 | acc: 97.99% | AUC: 0.973  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_small.pth
Epoch   2 | train: 0.2874 | val: 0.2167 | acc: 96.14% | AUC: 0.989  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_small.pth
Epoch   3 | train: 0.2078 | val: 0.1112 | acc: 98.11% | AUC: 0.996  | LR: 0.001
Epoch   4 | train: 0.1942 | val: 0.1243 | acc: 97.51% | AUC: 0.996  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_small.pth
Epoch   5 | train: 0.1873 | val: 0.1007 | acc: 98.45% | AUC: 0.996  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_small.pth
Epoch   6 | train: 0.1591 | val: 0.1542 | acc: 95.92% | AUC: 0.

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.099035,0.183867,97.632312,0.988936,0.988854,0.888889,1863,21,30,240
1,2,0.062801,0.105218,96.332405,0.992785,0.962845,0.966667,1814,70,9,261
2,3,0.097995,0.186478,96.609382,0.977072,0.977176,0.888476,1841,43,30,239
3,4,0.098791,0.178832,97.211896,0.983592,0.983006,0.895911,1851,32,28,241
4,5,0.128195,0.155325,94.196843,0.992399,0.936870,0.977695,1766,119,6,263
5,6,0.054150,0.099408,97.211896,0.991754,0.977164,0.936803,1840,43,17,252
6,7,0.083843,0.093577,97.353760,0.992723,0.977719,0.944238,1843,42,15,254
7,8,0.105868,0.226219,97.260910,0.980718,0.990451,0.847584,1867,18,41,228
8,9,0.106469,0.504116,96.425255,0.973250,0.975597,0.884758,1839,46,31,238
9,10,0.091266,0.126549,95.724907,0.988275,0.962294,0.921933,1812,71,21,248


In [3]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_medium import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 172,130
  -> ny bästa modell sparad till ../models/cnn_medium.pth
Epoch   0 | train: 3.1926 | val: 0.4294 | acc: 96.87% | AUC: 0.821  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_medium.pth
Epoch   1 | train: 0.5033 | val: 0.4425 | acc: 89.60% | AUC: 0.936  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_medium.pth
Epoch   2 | train: 0.3814 | val: 0.2229 | acc: 96.96% | AUC: 0.978  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_medium.pth
Epoch   3 | train: 0.2843 | val: 0.2086 | acc: 95.13% | AUC: 0.988  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_medium.pth
Epoch   4 | train: 0.2361 | val: 0.2070 | acc: 94.43% | AUC: 0.993  | LR: 0.001
Epoch   5 | train: 0.2322 | val: 0.2299 | acc: 94.65% | AUC: 0.983  | LR: 0.001
Epoch   6 | train: 0.2199 | val: 0.2704 | acc: 90.20% | AUC: 0.991  | LR: 0.001
  -> ny bästa modell sparad till .

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.074083,0.116643,97.818013,0.991745,0.983015,0.944444,1852,32,15,255
1,2,0.111630,0.123342,95.961003,0.990639,0.960722,0.951852,1810,74,13,257
2,3,0.085706,0.150263,97.073850,0.983599,0.980892,0.899628,1848,36,27,242
3,4,0.111073,0.155552,97.539461,0.985807,0.984085,0.914498,1855,30,23,246
4,5,0.085200,0.138537,95.171773,0.991651,0.949072,0.970260,1789,96,8,261
5,6,0.090134,0.086102,97.677659,0.993688,0.981423,0.944238,1849,35,15,254
6,7,0.109958,0.110276,97.770553,0.990914,0.986200,0.918216,1858,26,22,247
7,8,0.101400,0.209552,94.281729,0.974547,0.947396,0.910781,1783,99,24,245
8,9,0.108502,0.312828,94.475395,0.979345,0.945889,0.936803,1783,102,17,252
9,10,0.056400,0.144890,96.470042,0.988654,0.972399,0.910781,1832,52,24,245


In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_huge import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 1,505,826
  -> ny bästa modell sparad till ../models/cnn_huge.pth
Epoch   0 | train: 13.9208 | val: 0.3984 | acc: 96.11% | AUC: 0.855  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_huge.pth
Epoch   1 | train: 0.4606 | val: 0.2805 | acc: 96.65% | AUC: 0.924  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_huge.pth
Epoch   2 | train: 0.3953 | val: 0.2787 | acc: 95.86% | AUC: 0.952  | LR: 0.001
Epoch   3 | train: 0.4017 | val: 0.2602 | acc: 97.54% | AUC: 0.937  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_huge.pth
Epoch   4 | train: 0.3402 | val: 0.1731 | acc: 98.60% | AUC: 0.967  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_huge.pth
Epoch   5 | train: 0.2900 | val: 0.1565 | acc: 97.87% | AUC: 0.977  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_huge.pth
Epoch   6 | train: 0.2418 | val: 0.1367 | acc: 98.81% | AUC: 0.98

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.084865,0.110832,96.007428,0.990790,0.960191,0.959259,1809,75,11,259
1,2,0.080029,0.097213,96.702276,0.993102,0.967605,0.962963,1822,61,10,260
2,3,0.076526,0.141724,96.609382,0.987222,0.972399,0.921933,1832,52,21,248
3,4,0.088791,0.153521,96.048350,0.985120,0.961743,0.951673,1810,72,13,256
4,5,0.105475,0.168884,95.864312,0.985803,0.960170,0.947955,1808,75,14,255
5,6,0.101697,0.104931,96.703807,0.988960,0.970292,0.944238,1829,56,15,254
6,7,0.104429,0.132560,94.377323,0.987434,0.940520,0.966543,1771,112,9,260
7,8,0.071663,0.181953,97.539461,0.985947,0.985676,0.903346,1858,27,26,243
8,9,0.125679,0.341175,94.661096,0.977299,0.949072,0.929368,1789,96,19,250
9,10,0.722958,0.671676,88.759870,0.872115,1.000000,0.100372,1884,0,242,27
